# 🤖 Resume / Candidate Screening System
### Future Interns — Machine Learning Internship | Task 3
**Repository:** `FUTURE_ML_03`

---

## 📌 Project Overview
This notebook builds a complete **ML-powered Resume Screening System** that:
- 📄 Parses and cleans resume text
- 🔍 Extracts skills and matches them to job descriptions
- 🏆 Ranks candidates based on role fit score
- ⚠️ Identifies skill gaps for each candidate

**Tools Used:** Python, spaCy, NLTK, Scikit-learn, Pandas, Matplotlib, Seaborn

---

## ⚙️ Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install spacy nltk scikit-learn pandas matplotlib seaborn --quiet
!python -m spacy download en_core_web_sm --quiet

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)

print('✅ All libraries installed and ready!')

In [ ]:
# Core imports
import re
import string
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load spaCy model
nlp = spacy.load('en_core_web_sm')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

print('✅ All imports successful!')

---
## 📂 Step 2 — Sample Resume & Job Description Data
> *These are realistic sample resumes and job descriptions. You can replace them with real ones.*

In [ ]:
# ─────────────────────────────────────────────
#  SAMPLE RESUMES (5 candidates)
# ─────────────────────────────────────────────
resumes = [
    {
        "name": "Alice Johnson",
        "text": """
        Alice Johnson | alice@email.com | LinkedIn: linkedin.com/in/alice
        SUMMARY: Data Scientist with 2 years of experience in machine learning and data analysis.
        SKILLS: Python, Machine Learning, Deep Learning, TensorFlow, Scikit-learn, Pandas, NumPy,
                SQL, Data Visualization, Matplotlib, Seaborn, NLP, Git, Jupyter Notebook.
        EXPERIENCE:
          - Data Analyst at TechCorp (2022-2024): Built predictive models using scikit-learn.
            Performed EDA on large datasets. Created dashboards using Tableau.
        EDUCATION: B.Tech Computer Science, 2022. GPA: 8.5/10.
        PROJECTS: Sentiment Analysis using BERT. House Price Prediction using Random Forest.
        CERTIFICATIONS: Google Data Analytics, Coursera Machine Learning.
        """
    },
    {
        "name": "Bob Smith",
        "text": """
        Bob Smith | bob@email.com
        SUMMARY: Software developer with background in web development and basic Python scripting.
        SKILLS: JavaScript, React, Node.js, HTML, CSS, Python, MySQL, Git, REST APIs.
        EXPERIENCE:
          - Frontend Developer at WebAgency (2021-2023): Built responsive websites using React.
            Integrated REST APIs. Managed MySQL databases.
        EDUCATION: B.Sc Information Technology, 2021.
        PROJECTS: E-commerce website, Portfolio builder app.
        """
    },
    {
        "name": "Carol Lee",
        "text": """
        Carol Lee | carol@email.com
        SUMMARY: Machine Learning Engineer with strong statistical background and 3 years experience.
        SKILLS: Python, R, Machine Learning, Deep Learning, PyTorch, TensorFlow, Scikit-learn,
                Statistics, Pandas, NumPy, SQL, MongoDB, Docker, Kubernetes, MLOps, Git, NLP,
                Computer Vision, Data Pipelines, Feature Engineering, A/B Testing.
        EXPERIENCE:
          - ML Engineer at AI Startup (2021-2024): Deployed ML models to production using Docker.
            Built NLP pipelines. Optimized deep learning models with PyTorch.
          - Research Intern at University Lab (2020): Published paper on image classification.
        EDUCATION: M.Sc Data Science, 2021. B.Sc Statistics, 2019.
        PROJECTS: Real-time object detection system. Customer churn prediction pipeline.
        CERTIFICATIONS: AWS ML Specialty, TensorFlow Developer Certificate.
        """
    },
    {
        "name": "David Patel",
        "text": """
        David Patel | david@email.com
        SUMMARY: Recent graduate with internship experience in data science.
        SKILLS: Python, Pandas, NumPy, Scikit-learn, SQL, Matplotlib, Git, Excel, Power BI.
        EXPERIENCE:
          - Data Science Intern at DataCo (2023): Performed data cleaning and EDA.
            Built classification models. Created reports using Power BI.
        EDUCATION: B.Tech Computer Science, 2023. GPA: 7.8/10.
        PROJECTS: Titanic Survival Prediction. Customer Segmentation using K-Means.
        CERTIFICATIONS: IBM Data Science Professional Certificate.
        """
    },
    {
        "name": "Eva Rodriguez",
        "text": """
        Eva Rodriguez | eva@email.com
        SUMMARY: Business analyst transitioning into data science with strong analytical skills.
        SKILLS: Excel, Power BI, SQL, Python basics, Statistics, Business Analysis,
                Data Visualization, Tableau, Communication, Project Management.
        EXPERIENCE:
          - Business Analyst at ConsultFirm (2020-2024): Analyzed business data using Excel.
            Created dashboards in Power BI. Wrote SQL queries for reporting.
        EDUCATION: MBA Business Analytics, 2020.
        PROJECTS: Sales performance dashboard. Market trend analysis report.
        """
    }
]

# ─────────────────────────────────────────────
#  JOB DESCRIPTION
# ─────────────────────────────────────────────
job_description = """
Job Title: Machine Learning Engineer

We are looking for a skilled Machine Learning Engineer to join our AI team.

Required Skills:
- Python programming (mandatory)
- Machine Learning algorithms (Scikit-learn, XGBoost)
- Deep Learning frameworks (TensorFlow or PyTorch)
- Data manipulation with Pandas and NumPy
- SQL for data querying
- Natural Language Processing (NLP)
- Git version control
- Data visualization (Matplotlib, Seaborn, Tableau)
- Statistics and probability
- Feature Engineering

Nice to Have:
- Docker, Kubernetes, MLOps
- Cloud platforms (AWS, GCP, Azure)
- Computer Vision experience
- Published research or open source contributions

Qualifications:
- B.Tech/M.Sc in Computer Science, Data Science or related field
- 1-3 years of relevant experience
"""

# Required skills list (ground truth for gap analysis)
required_skills = [
    'python', 'machine learning', 'deep learning', 'tensorflow', 'pytorch',
    'scikit-learn', 'pandas', 'numpy', 'sql', 'nlp', 'git',
    'matplotlib', 'seaborn', 'statistics', 'feature engineering',
    'docker', 'kubernetes', 'mlops', 'computer vision'
]

print(f'✅ Loaded {len(resumes)} candidate resumes')
print(f'✅ Job Description: Machine Learning Engineer')
print(f'✅ Total required skills to match: {len(required_skills)}')

---
## 🧹 Step 3 — Text Cleaning & Preprocessing

In [ ]:
def clean_text(text):
    """Clean and normalize resume/JD text."""
    # Lowercase
    text = text.lower()
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove special characters and digits
    text = re.sub(r'[^a-z\s\-]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def preprocess_text(text):
    """Tokenize, remove stopwords, and lemmatize."""
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)


# Apply preprocessing
cleaned_resumes = []
for r in resumes:
    cleaned_resumes.append({
        'name': r['name'],
        'original': r['text'],
        'cleaned': preprocess_text(r['text'])
    })

cleaned_jd = preprocess_text(job_description)

print('✅ Text cleaning complete!\n')
print('📄 Sample cleaned resume (Alice Johnson):')
print('─' * 60)
print(cleaned_resumes[0]['cleaned'][:300], '...')

---
## 🔍 Step 4 — Skill Extraction using spaCy & Keyword Matching

In [ ]:
def extract_skills(text, skill_list):
    """Extract matching skills from resume text."""
    text_lower = text.lower()
    found_skills = []
    for skill in skill_list:
        # Handle multi-word skills and variations
        skill_variants = [
            skill,
            skill.replace('-', ' '),
            skill.replace(' ', '-'),
            skill.replace(' ', '')
        ]
        for variant in skill_variants:
            if re.search(r'\b' + re.escape(variant) + r'\b', text_lower):
                found_skills.append(skill)
                break
    return list(set(found_skills))


def identify_skill_gaps(found_skills, required_skills):
    """Return skills that are required but missing from resume."""
    return [s for s in required_skills if s not in found_skills]


# Extract skills for each candidate
for candidate in cleaned_resumes:
    candidate['skills_found'] = extract_skills(candidate['original'], required_skills)
    candidate['skill_gaps']   = identify_skill_gaps(candidate['skills_found'], required_skills)
    candidate['skill_count']  = len(candidate['skills_found'])

print('✅ Skill extraction complete!\n')
for c in cleaned_resumes:
    print(f"👤 {c['name']}")
    print(f"   ✅ Found  ({c['skill_count']}): {', '.join(c['skills_found']) if c['skills_found'] else 'None'}")
    print(f"   ❌ Missing ({len(c['skill_gaps'])}): {', '.join(c['skill_gaps']) if c['skill_gaps'] else 'None'}")
    print()

---
## 📊 Step 5 — TF-IDF Similarity Scoring (ML-based Matching)

In [ ]:
# Build TF-IDF matrix
all_docs = [cleaned_jd] + [c['cleaned'] for c in cleaned_resumes]

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # Unigrams + Bigrams for better matching
    max_features=1000,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(all_docs)

# Cosine similarity: JD (index 0) vs each resume
jd_vector       = tfidf_matrix[0]
resume_vectors  = tfidf_matrix[1:]
similarity_scores = cosine_similarity(jd_vector, resume_vectors)[0]

# Compute composite score (TF-IDF + skill match ratio)
for i, candidate in enumerate(cleaned_resumes):
    tfidf_score      = round(similarity_scores[i] * 100, 2)
    skill_ratio      = candidate['skill_count'] / len(required_skills)
    composite_score  = round((tfidf_score * 0.6) + (skill_ratio * 100 * 0.4), 2)

    candidate['tfidf_score']     = tfidf_score
    candidate['skill_ratio']     = round(skill_ratio * 100, 2)
    candidate['composite_score'] = composite_score

print('✅ Scoring complete!\n')
print(f'{"Candidate":<20} {"TF-IDF":<12} {"Skill Match":<14} {"Composite"}')
print('─' * 60)
for c in cleaned_resumes:
    print(f"{c['name']:<20} {c['tfidf_score']:<12} {c['skill_ratio']:<14} {c['composite_score']}")

---
## 🏆 Step 6 — Candidate Ranking

In [ ]:
# Build ranked DataFrame
df = pd.DataFrame([{
    'Rank':            0,
    'Candidate':       c['name'],
    'TF-IDF Score':    c['tfidf_score'],
    'Skill Match (%)': c['skill_ratio'],
    'Composite Score': c['composite_score'],
    'Skills Found':    c['skill_count'],
    'Skills Missing':  len(c['skill_gaps']),
    'Skills Found List':   ', '.join(sorted(c['skills_found'])),
    'Skill Gaps':          ', '.join(sorted(c['skill_gaps'])) if c['skill_gaps'] else 'None'
} for c in cleaned_resumes])

df = df.sort_values('Composite Score', ascending=False).reset_index(drop=True)
df['Rank'] = range(1, len(df) + 1)

# Add suitability label
def label(score):
    if score >= 60: return '🟢 Highly Suitable'
    elif score >= 35: return '🟡 Moderately Suitable'
    else: return '🔴 Not Suitable'

df['Suitability'] = df['Composite Score'].apply(label)

print('=' * 70)
print('           🏆 CANDIDATE RANKING — ML ENGINEER ROLE')
print('=' * 70)
display_cols = ['Rank', 'Candidate', 'Composite Score', 'Skill Match (%)', 'Suitability']
print(df[display_cols].to_string(index=False))
print('=' * 70)

---
## 📉 Step 7 — Skill Gap Analysis per Candidate

In [ ]:
print('=' * 70)
print('         🔍 DETAILED SKILL GAP ANALYSIS')
print('=' * 70)

for _, row in df.iterrows():
    print(f"\n#{row['Rank']} — {row['Candidate']} | Score: {row['Composite Score']} | {row['Suitability']}")
    print(f"   ✅ Skills Present  : {row['Skills Found List']}")
    print(f"   ❌ Skills Missing  : {row['Skill Gaps']}")
    print(f"   📊 Match Coverage  : {row['Skills Found']} / {len(required_skills)} required skills")

---
## 📈 Step 8 — Visualizations

In [ ]:
# ── Chart 1: Composite Score Ranking ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

colors = ['#2ecc71' if s >= 60 else '#f39c12' if s >= 35 else '#e74c3c'
          for s in df['Composite Score']]

bars = ax.barh(df['Candidate'], df['Composite Score'], color=colors, edgecolor='white', height=0.6)

for bar, score in zip(bars, df['Composite Score']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score}', va='center', fontweight='bold', fontsize=11)

ax.set_xlabel('Composite Score (out of 100)', fontsize=12)
ax.set_title('🏆 Candidate Ranking — Machine Learning Engineer Role', fontsize=14, fontweight='bold', pad=15)
ax.set_xlim(0, 105)
ax.invert_yaxis()

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Highly Suitable (≥60)'),
    mpatches.Patch(color='#f39c12', label='Moderately Suitable (35-59)'),
    mpatches.Patch(color='#e74c3c', label='Not Suitable (<35)')
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('candidate_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as candidate_ranking.png')

In [ ]:
# ── Chart 2: Skills Found vs Missing ──────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df))
width = 0.35

bars1 = ax.bar(x - width/2, df['Skills Found'],  width, label='✅ Skills Found',   color='#27ae60', alpha=0.85)
bars2 = ax.bar(x + width/2, df['Skills Missing'], width, label='❌ Skills Missing', color='#e74c3c', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df['Candidate'], rotation=15, ha='right')
ax.set_ylabel('Number of Skills', fontsize=12)
ax.set_title('🔍 Skills Found vs Skills Missing per Candidate', fontsize=14, fontweight='bold', pad=15)
ax.legend(fontsize=11)
ax.set_ylim(0, len(required_skills) + 3)
plt.tight_layout()
plt.savefig('skills_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as skills_comparison.png')

In [ ]:
# ── Chart 3: Skill Coverage Heatmap ───────────────────────────────────
# Build binary matrix: candidate x skill
candidates_ordered = df['Candidate'].tolist()
skill_matrix = []

for name in candidates_ordered:
    cand = next(c for c in cleaned_resumes if c['name'] == name)
    row  = [1 if s in cand['skills_found'] else 0 for s in required_skills]
    skill_matrix.append(row)

heatmap_df = pd.DataFrame(skill_matrix, index=candidates_ordered, columns=required_skills)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    heatmap_df,
    annot=True, fmt='d', cmap='RdYlGn',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': '0 = Missing | 1 = Present'},
    ax=ax
)
ax.set_title('📊 Skill Coverage Heatmap — All Candidates', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Required Skills', fontsize=11)
ax.set_ylabel('Candidate', fontsize=11)
plt.xticks(rotation=35, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig('skill_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as skill_heatmap.png')

In [ ]:
# ── Chart 4: Score Breakdown (TF-IDF vs Skill Match) ──────────────────
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(df))
width = 0.3

ax.bar(x - width,     df['TF-IDF Score'],    width, label='TF-IDF Score (60%)',    color='#3498db', alpha=0.85)
ax.bar(x,             df['Skill Match (%)'], width, label='Skill Match (40%)',      color='#9b59b6', alpha=0.85)
ax.bar(x + width,     df['Composite Score'], width, label='Composite Score (Final)',color='#e67e22', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df['Candidate'], rotation=15, ha='right')
ax.set_ylabel('Score', fontsize=12)
ax.set_title('📉 Score Breakdown: TF-IDF vs Skill Match vs Composite', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('score_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as score_breakdown.png')

---
## 💾 Step 9 — Export Results to CSV

In [ ]:
# Export full results
export_cols = ['Rank', 'Candidate', 'Composite Score', 'TF-IDF Score',
               'Skill Match (%)', 'Skills Found', 'Skills Missing', 'Suitability',
               'Skills Found List', 'Skill Gaps']

df[export_cols].to_csv('resume_screening_results.csv', index=False)

print('✅ Results exported to resume_screening_results.csv')
print()
print('📁 Files generated:')
print('   • resume_screening_results.csv')
print('   • candidate_ranking.png')
print('   • skills_comparison.png')
print('   • skill_heatmap.png')
print('   • score_breakdown.png')

---
## ✅ Step 10 — Final Summary & Recommendation

In [ ]:
top_candidate   = df.iloc[0]
shortlisted     = df[df['Suitability'].str.contains('Highly')]
moderate        = df[df['Suitability'].str.contains('Moderately')]
not_suitable    = df[df['Suitability'].str.contains('Not')]

print('=' * 70)
print('           ✅ FINAL SCREENING SUMMARY')
print('=' * 70)
print(f"\n🎯 Job Role           : Machine Learning Engineer")
print(f"👥 Total Candidates   : {len(df)}")
print(f"🏅 Required Skills    : {len(required_skills)}")
print()
print(f"🟢 Highly Suitable    : {len(shortlisted)} candidate(s) — {', '.join(shortlisted['Candidate'].tolist())}")
print(f"🟡 Moderately Suitable: {len(moderate)} candidate(s) — {', '.join(moderate['Candidate'].tolist())}")
print(f"🔴 Not Suitable       : {len(not_suitable)} candidate(s) — {', '.join(not_suitable['Candidate'].tolist())}")
print()
print(f"🏆 TOP RECOMMENDATION : {top_candidate['Candidate']}")
print(f"   Composite Score    : {top_candidate['Composite Score']} / 100")
print(f"   Skill Match        : {top_candidate['Skill Match (%)']}%")
print(f"   Skill Gaps         : {top_candidate['Skill Gaps']}")
print()
print('=' * 70)
print('  ✅ Resume Screening System — Task 3 Complete!')
print('  📁 GitHub Repo: FUTURE_ML_03')
print('=' * 70)

---
## 📝 Conclusion

This **Resume / Candidate Screening System** successfully demonstrates:

| Step | What Was Done |
|------|---------------|
| 1 | Installed and imported all ML/NLP libraries |
| 2 | Created realistic resume and job description data |
| 3 | Cleaned and preprocessed text (stopwords, lemmatization) |
| 4 | Extracted and matched skills using keyword + spaCy |
| 5 | Scored candidates using TF-IDF + Cosine Similarity |
| 6 | Ranked candidates by composite score |
| 7 | Identified skill gaps for each candidate |
| 8 | Visualized results with 4 charts |
| 9 | Exported results to CSV |
| 10 | Generated final hiring recommendation |

**Key Learning:** Combining TF-IDF similarity with explicit skill matching gives a more reliable ranking than either method alone.

---
*Future Interns — Machine Learning Internship | Task 3 | Repository: FUTURE_ML_03*